write a python notebook (not in style of functions but in code blocks to be run sequentially) to import relevant libraries, load a csv data set (labelled: target = last column called class --> two categories: 1 = real, 2 = fake), the first column of the data set is the file path so we might have to preprocess that, load a random forest model. split the data set into x, y, train, and test. Define a very extensive hyperparameter grid for the random forest classifier and then train and test the model on the dataset using the hyperparameter grid (verbose = 1 and textual evaluation results printed along with the hyperparameters used to train). save the best performing model locally using joblib and then beautifully print all possible metrics for evaluation (e.g., accuracy, precision, f1 score, recall, roc/auc curve, confusion matrices) for the best model.

In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# 2. Load the dataset
# Replace 'your_data.csv' with the actual filename
df = pd.read_csv('../dataset/01_feature_csv/Inception_TEST_real_fake_hard.csv')

# Preview the data
df.head()


,image_path,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,class
0,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.112439,0.058074,0.019880,0.511245,0.119854,0.328928,0.078603,0.270154,0.387435,...,0.000643,0.002497,1.751615,0.489238,1.382552,0.482872,0.147175,0.045142,0.372681,1
1,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.180687,0.702820,0.175272,0.392263,0.153603,0.385228,0.099636,0.084808,0.093171,...,0.746194,0.004088,1.233643,0.437269,0.832287,0.083574,0.132807,0.153932,0.836535,1
2,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.132383,0.645224,0.072233,0.341254,0.248971,0.172578,0.098041,0.083010,0.283894,...,0.709060,0.040206,1.793995,0.064851,0.872768,0.006697,0.243913,0.245824,0.463083,1
3,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.107133,0.267212,0.171197,0.173169,0.195927,0.355011,0.581593,0.070428,0.083302,...,0.059308,0.061095,1.875852,0.223341,0.786362,0.037105,0.213425,0.212645,0.668771,1
4,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.184027,0.607342,0.126969,0.546473,0.359387,0.220068,0.267449,0.512553,0.096198,...,0.770838,0.095556,1.807522,0.282321,0.495637,0.203213,0.144256,0.436093,1.451752,1


In [3]:
# 3. Preprocess: Remove first column (file path), separate features and labels
X = df.iloc[:, 1:-1] # All columns except the first (file path) and last (class)
y = df['class']      # Target column

In [4]:
X.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,0.112439,0.058074,0.019880,0.511245,0.119854,0.328928,0.078603,0.270154,0.387435,0.105396,...,0.466098,0.000643,0.002497,1.751615,0.489238,1.382552,0.482872,0.147175,0.045142,0.372681
1,0.180687,0.702820,0.175272,0.392263,0.153603,0.385228,0.099636,0.084808,0.093171,0.076507,...,0.307027,0.746194,0.004088,1.233643,0.437269,0.832287,0.083574,0.132807,0.153932,0.836535
2,0.132383,0.645224,0.072233,0.341254,0.248971,0.172578,0.098041,0.083010,0.283894,0.118709,...,0.322262,0.709060,0.040206,1.793995,0.064851,0.872768,0.006697,0.243913,0.245824,0.463083
3,0.107133,0.267212,0.171197,0.173169,0.195927,0.355011,0.581593,0.070428,0.083302,0.570711,...,0.027984,0.059308,0.061095,1.875852,0.223341,0.786362,0.037105,0.213425,0.212645,0.668771
4,0.184027,0.607342,0.126969,0.546473,0.359387,0.220068,0.267449,0.512553,0.096198,0.386062,...,0.123806,0.770838,0.095556,1.807522,0.282321,0.495637,0.203213,0.144256,0.436093,1.451752


In [5]:
y.value_counts()

class
2    700
1    589
Name: count, dtype: int64

In [6]:
# 4. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=420, stratify=y
)
print(f'Training size: {X_train.shape}, Test size: {X_test.shape}')


Training size: (1031, 2048), Test size: (258, 2048)


In [8]:
# 5. Define an extensive hyperparameter grid
from sklearn.model_selection import ParameterGrid
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 10, 30, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced']
}
hyperparams_list = list(ParameterGrid(param_grid))
print(f"Total hyperparameter combinations: {len(hyperparams_list)}")

Total hyperparameter combinations: 1296


In [ ]:
# 6. Train and test each combination, store the model with the best test F1 score
best_f1 = 0
best_model = None
best_params = None
results_log = []

for i, params in enumerate(hyperparams_list):
    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, pos_label=1, average='binary')

    results_log.append({
        'index': i,
        **params,
        'f1_test': f1,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, pos_label=1),
        'recall': recall_score(y_test, y_pred, pos_label=1)
    })

    print(f"Run {i+1}/{len(hyperparams_list)} | Hyperparams: {params} | Test F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_params = params

print('-'*60)
print("Best Params on Test Set:", best_params)
print(f"Best Test F1: {best_f1:.4f}")


Run 1/1296 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100} | Test F1: 0.9793
Run 2/1296 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} | Test F1: 0.9707
Run 3/1296 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500} | Test F1: 0.9620
Run 4/1296 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100} | Test F1: 0.9623
Run 5/1296 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200} | Test F1: 0.9540
Run 6/1296 | Hy

In [ ]:
# 7. Save the best model
joblib.dump(best_model, 'best_rf_model_on_test.joblib')
print('Best model saved as best_rf_model_on_test.joblib')

In [ ]:
# 8. Evaluate best model thoroughly on the test set
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:,1]
print('Classification Report:')
print(classification_report(y_test, y_pred))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, pos_label=1):.4f}')
print(f'Recall: {recall_score(y_test, y_pred, pos_label=1):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred, pos_label=1):.4f}')

roc_auc = roc_auc_score(y_test, y_proba)
print(f'ROC AUC Score: {roc_auc:.4f}')


In [ ]:
# 9. Plot confusion matrix
plt.figure(figsize=(5,5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
# 10. Plot ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba, pos_label=1)
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, color='blue', label=f'AUC={roc_auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid()
plt.show()
